# Setup From GitHub

Set `REPO_URL` to your GitHub repo. These notebooks avoid reinstalling PyTorch because Kaggle already ships a CUDA-compatible build.

In [ ]:
REPO_URL = "https://github.com/ahmad903501/qwen-dpo-alignment.git"
BRANCH = "main"
REPO_DIR = "/kaggle/working/qwen-dpo-alignment"


In [ ]:
from pathlib import Path

repo_path = Path(REPO_DIR)
if not repo_path.exists():
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull --ff-only origin {BRANCH}

%cd {REPO_DIR}
!pwd


In [ ]:
# Kaggle may preinstall an old torchao that PEFT refuses when loading LoRA adapters.
# We do not use torchao quantization in these notebooks, so remove it first.
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors
!python -m pip install -q -e . --no-deps

import torch, transformers, datasets
print("torch", torch.__version__)
print("transformers", transformers.__version__)
print("datasets", datasets.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA devices:", torch.cuda.device_count())
!nvidia-smi

import os
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
except Exception as exc:
    print(f"No Kaggle HF_TOKEN secret loaded: {exc}")

if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login
    login(token=os.environ["HF_TOKEN"])



# Generation Only

Use this notebook to generate AlpacaEval-subset outputs from base, SFT, and DPO models. If the checkpoints were trained in another Kaggle notebook, attach that notebook's outputs as an input dataset and set `SFT_MODEL_PATH` / `DPO_MODEL_PATH` accordingly.

In [ ]:
SUBSET_SIZE = 300
SEED = 42
MODEL_NAME = "Qwen/Qwen2.5-0.5B"
MODEL_RUN_NAME = MODEL_NAME.split("/")[-1].lower().replace(".", "-")

BASE_MODEL_PATH = MODEL_NAME
SFT_MODEL_PATH = f"outputs/sft-{MODEL_RUN_NAME}/best"
DPO_MODEL_PATH = f"outputs/dpo-{MODEL_RUN_NAME}-ultrafeedback/best"

# Use "chat_template" for instruction-tuned chat models such as Llama-3.2-1B-Instruct.
BASE_PROMPT_FORMAT = "plain"
SFT_PROMPT_FORMAT = "plain"
DPO_PROMPT_FORMAT = "plain"
REPETITION_PENALTY = 1.15
NO_REPEAT_NGRAM_SIZE = 4

BASE_GENERATOR_NAME = f"{MODEL_RUN_NAME}-base"
SFT_GENERATOR_NAME = f"{MODEL_RUN_NAME}-sft"
DPO_GENERATOR_NAME = f"{MODEL_RUN_NAME}-dpo"
ARCHIVE_BASENAME = f"{MODEL_RUN_NAME}_alignment_eval_outputs_{SUBSET_SIZE}_seed{SEED}"

# If using a separate Kaggle notebook with attached outputs, set SFT_MODEL_PATH / DPO_MODEL_PATH
# to /kaggle/input/... checkpoint paths while keeping MODEL_RUN_NAME as the display/run label.

SUBSET_PATH = f"data/alpaca_eval/alpacaeval_{SUBSET_SIZE}_seed{SEED}.json"

## Create AlpacaEval Subset

In [ ]:
!python scripts/make_alpaca_subset.py \
  --size {SUBSET_SIZE} \
  --seed {SEED} \
  --output_dir data/alpaca_eval


## Sanity-Check a Few Generations

Before generating the full subset, run 2-5 prompts and inspect the outputs. This catches path, tokenizer, prompt-format, and checkpoint issues early.


In [ ]:
SANITY_SIZE = 5
!python scripts/generate_alpaca.py \
  --model_name_or_path {SFT_MODEL_PATH} \
  --subset_path {SUBSET_PATH} \
  --output_path outputs/eval/sanity_sft_{SANITY_SIZE}.json \
  --generator_name {SFT_GENERATOR_NAME}-sanity \
  --batch_size 2 \
  --max_new_tokens 256 \
  --limit {SANITY_SIZE} \
  --device cuda:0

import json
with open(f"outputs/eval/sanity_sft_{SANITY_SIZE}.json") as f:
    sanity_outputs = json.load(f)
for row in sanity_outputs:
    print("=" * 80)
    print("PROMPT:", row["instruction"][:500])
    print("OUTPUT:", row["output"][:1000])


## Two-GPU Sharded Generation

Each model generation is split into two interleaved prompt shards. Kaggle T4x2 can run one shard on `cuda:0` and the other on `cuda:1`, then the notebook merges the shard files back into the original subset order.


In [ ]:
import json, subprocess, sys
from pathlib import Path

def merge_generation_shards(subset_path, shard_paths, output_path):
    with open(subset_path) as f:
        subset = json.load(f)
    by_id = {}
    for shard_path in shard_paths:
        with open(shard_path) as f:
            for row in json.load(f):
                by_id[row.get("id")] = row
    merged = [by_id[row.get("id")] for row in subset if row.get("id") in by_id]
    if len(merged) != len(subset):
        raise RuntimeError(f"Merged {len(merged)} rows, expected {len(subset)}")
    with open(output_path, "w") as f:
        json.dump(merged, f, indent=2)
    print(f"Merged {len(merged)} generations -> {output_path}")

def generate_on_two_gpus(model_path, generator_name, output_path, batch_size=8, max_new_tokens=512, prompt_format="plain", repetition_penalty=1.15, no_repeat_ngram_size=4):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    shard_paths = [output_path.with_suffix(f".shard{i}.json") for i in range(2)]
    procs = []
    for shard_idx, shard_path in enumerate(shard_paths):
        cmd = [
            sys.executable, "scripts/generate_alpaca.py",
            "--model_name_or_path", str(model_path),
            "--subset_path", str(SUBSET_PATH),
            "--output_path", str(shard_path),
            "--generator_name", generator_name,
            "--batch_size", str(batch_size),
            "--max_new_tokens", str(max_new_tokens),
            "--prompt_format", prompt_format,
            "--repetition_penalty", str(repetition_penalty),
            "--no_repeat_ngram_size", str(no_repeat_ngram_size),
            "--device", f"cuda:{shard_idx}",
            "--num_shards", "2",
            "--shard_index", str(shard_idx),
        ]
        print("Launching:", " ".join(cmd))
        procs.append(subprocess.Popen(cmd))
    failed = [p.wait() for p in procs]
    if any(code != 0 for code in failed):
        raise RuntimeError(f"Generation shard failed with exit codes: {failed}")
    merge_generation_shards(SUBSET_PATH, shard_paths, output_path)


## Generate Base, SFT, and DPO Outputs


In [ ]:
generate_on_two_gpus(BASE_MODEL_PATH, BASE_GENERATOR_NAME, f"outputs/eval/base_alpacaeval_{SUBSET_SIZE}.json", prompt_format=BASE_PROMPT_FORMAT, repetition_penalty=REPETITION_PENALTY, no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE)
generate_on_two_gpus(SFT_MODEL_PATH, SFT_GENERATOR_NAME, f"outputs/eval/sft_alpacaeval_{SUBSET_SIZE}.json", prompt_format=SFT_PROMPT_FORMAT, repetition_penalty=REPETITION_PENALTY, no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE)
generate_on_two_gpus(DPO_MODEL_PATH, DPO_GENERATOR_NAME, f"outputs/eval/dpo_alpacaeval_{SUBSET_SIZE}.json", prompt_format=DPO_PROMPT_FORMAT, repetition_penalty=REPETITION_PENALTY, no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE)


## Build Pairwise Files For Evaluation

In [ ]:
!python scripts/build_pairwise_eval.py \
  --a_outputs outputs/eval/dpo_alpacaeval_{SUBSET_SIZE}.json \
  --b_outputs outputs/eval/sft_alpacaeval_{SUBSET_SIZE}.json \
  --a_name dpo \
  --b_name sft \
  --output_path outputs/eval/dpo_vs_sft_pairs.jsonl

!python scripts/build_pairwise_eval.py \
  --a_outputs outputs/eval/dpo_alpacaeval_{SUBSET_SIZE}.json \
  --b_outputs outputs/eval/base_alpacaeval_{SUBSET_SIZE}.json \
  --a_name dpo \
  --b_name base \
  --output_path outputs/eval/dpo_vs_base_pairs.jsonl

!python scripts/build_pairwise_eval.py \
  --a_outputs outputs/eval/sft_alpacaeval_{SUBSET_SIZE}.json \
  --b_outputs outputs/eval/base_alpacaeval_{SUBSET_SIZE}.json \
  --a_name sft \
  --b_name base \
  --output_path outputs/eval/sft_vs_base_pairs.jsonl

!python scripts/build_pairwise_eval.py \
  --a_outputs outputs/eval/dpo_alpacaeval_{SUBSET_SIZE}.json \
  --b_from_reference \
  --a_name dpo \
  --b_name gpt_reference \
  --output_path outputs/eval/dpo_vs_reference_pairs.jsonl


## Check Generation Artifacts

In [ ]:
!find outputs/eval -maxdepth 1 -type f | sort


## Archive Evaluation Outputs

This creates a consistently named zip that the evaluation notebook knows how to consume. The archive contains an `eval/` directory with generations, pairwise files, shard files, and sanity-check outputs.


In [ ]:
import shutil
from pathlib import Path

ARCHIVE_PATH = f"/kaggle/working/{ARCHIVE_BASENAME}.zip"

# Archive with eval/ as the top-level directory.
shutil.make_archive(
    f"/kaggle/working/{ARCHIVE_BASENAME}",
    "zip",
    "/kaggle/working/qwen-dpo-alignment/outputs",
    "eval",
)

print(f"Created {ARCHIVE_PATH}")
!ls -lh {ARCHIVE_PATH}
